# Setup (run once per cluster)

Install dependencies from `pyproject.toml`. Run this notebook **once** when you first attach a cluster; packages are then available to **all** notebooks (engineering, analytics, science).

In [ ]:
from __future__ import annotations

import os
import sys
import subprocess
from pathlib import Path

def _find_root_from_cwd(max_parents: int = 8) -> Path | None:
    root = Path.cwd()
    for _ in range(max_parents + 1):
        if (root / "pyproject.toml").exists():
            return root
        root = root.parent
    return None


def _find_root_from_notebook_path() -> Path | None:
    if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
        return None

    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()  # type: ignore[name-defined]
        nb_path = ctx.notebookPath().get()  # e.g. /Repos/<user>/pokedata/notebooks/setup
        local_nb_path = Path("/Workspace") / nb_path.lstrip("/")
        root = local_nb_path
        for _ in range(12):
            if (root / "pyproject.toml").exists():
                return root
            root = root.parent
    except Exception:
        return None

    return None


root = _find_root_from_cwd() or _find_root_from_notebook_path()
if root is None:
    raise RuntimeError(
        "Could not locate pyproject.toml. Run this notebook from inside the repo (Databricks Repos)."
    )

print(f"Installing from: {root}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(root), "-q"])
print("Install complete.")

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    print("Restarting Python to pick up installed packages...")
    dbutils.library.restartPython()